**Objetivo**


Que una caída de luz no obligue a rehacer análisis forense manual sobre disco. El sistema debe poder responder siempre
a estas preguntas:

- cuál era el universo exacto de esta corrida
- qué ticker,date se esperaba descargar
- qué ticker,date está descargado y validado
- qué está descargado pero no validado
- qué falló
- qué queda pendiente
- qué se puede relanzar sin riesgo de saltarse nada ni duplicar lógica

## Arquitectura propuesta

### 1. Separar 4 fases de forma dura

1. Universe Build
2. Task Build
3. Download
4. Validate + Coverage + Reconcile

Ahora mismo esas cosas están mezcladas entre notebooks, runbooks y agentes. Hay que convertirlas en productos
explícitos.

———

### 2. Un único Run Manifest como fuente de verdad

Cada corrida debe tener un directorio con un manifiesto inmutable, por ejemplo:

- run_config.json
- universe_snapshot.parquet
- task_manifest.parquet
- task_manifest.csv opcional
- state/task_status.parquet
- state/file_index.parquet
- state/retry_queue.parquet
- reports/*.json

El centro del sistema debe ser task_manifest.parquet.

Columnas mínimas:

- run_id
- ticker
- date
- task_key = ticker|date
- expected_relpath
- universe_source
- lifecycle_source
- date_policy
- created_at_utc

Y muy importante:

- manifest_hash
- schema_version

Eso evita que el resume dependa de CSVs que cambian o notebooks que regeneran cosas sin control.

———

### 3. El universo se congela antes de descargar

No volvería a construir tickers_quotes_prod.csv dentro de un notebook de Agent03.

Haría un paso explícito:

- build_quotes_run_inputs.py

Entradas:

- universo refinado final
- lifecycle oficial
- calendario oficial XNYS
- date_from
- date_to

Salidas:

- universe_snapshot.parquet
- tickers_quotes_prod.parquet
- tasks_quotes_prod.parquet
- tasks_quotes_prod_meta.json

Reglas duras:

- usar calendario oficial, no pd.bdate_range
- recortar por rango explícito del run
- no generar tareas fuera de ventana
- no depender de celdas manuales

———

### 4. Agent01 no debe usar download_events_current.csv como verdad principal

Eso fue útil, pero es frágil.

La verdad debe ser una tabla de estado por task_key, por ejemplo:

- state/task_status.parquet

Con una fila por tarea y columnas como:

- task_key
- ticker
- date
- expected_relpath
- download_status
- validation_status
- coverage_status
- attempt_count
- last_attempt_utc
- last_success_utc
- last_error_code
- last_error_detail
- rows_written
- file_size_bytes
- file_sha256
- writer_run_id
- validator_run_id

Estados de descarga claros:

- PENDING
- DOWNLOADING
- DOWNLOADED_NONEMPTY
- DOWNLOADED_EMPTY_UNCONFIRMED
- EMPTY_CONFIRMED
- DOWNLOAD_FAILED
- DOWNLOAD_PARTIAL

Y separados de validación:

- NOT_VALIDATED
- VALID_PASS
- VALID_SOFT_FAIL
- VALID_HARD_FAIL

Así se evita mezclar física con calidad.

———

### 5. Escritura de Agent01 con publicación de 2 fases

No basta con .tmp -> replace.

Yo haría:

1. descargar a staging:
    - staging/<task_key>.parquet.tmp
2. validar estructura mínima local inmediatamente
3. si pasa:
    - mover a quotes_root/.../quotes.parquet
4. registrar commit en task_status

Y además generar un sidecar por archivo:

- quotes.parquet.meta.json

Con:

- task_key
- rows
- sha256
- file_size
- written_at_utc
- api_window_start
- api_window_end
- pages_fetched
- api_status_final

Eso hace el file auditable sin releer todo el history.

———

### 6. DOWNLOADED_EMPTY no debe ser terminal en resume

Este es uno de los fallos más claros del diseño actual.

Regla nueva:

- DOWNLOADED_EMPTY_UNCONFIRMED no se excluye en resume
- solo EMPTY_CONFIRMED excluye

Y EMPTY_CONFIRMED solo se consigue cuando:

- se ha reintentado N veces
- idealmente en runs distintos o ventanas temporales distintas
- y además la fecha está en calendario oficial del mercado

———

### 7. Reconciliación de disco automática, no forense manual

Debe existir un comando único:

- reconcile_run_state.py

Que compare:

- task_manifest
- task_status
- disco real
- sidecars
- outputs de validación

Y emita:

- missing_on_disk
- on_disk_not_registered
- registered_but_invalid
- downloaded_not_validated
- validated_not_in_manifest

Y sobre todo reconstruya un único:

- retry_manifest.parquet

No más missing_only_final.csv, missing_only_final_v2.csv, etc. Eso es síntoma de proceso no determinista.

———

### 8. Agent02 debe validar por task_manifest, no por escaneo libre de todo D:\quotes

Hoy Agent02 escanea el root completo y eso mezcla runs, residuos y otras campañas.

Nuevo diseño:

- Agent02 recibe:
    - run_id
    - task_manifest
    - task_status
- solo valida los expected_relpath del run
- opcionalmente hace un auditor extra de “archivos huérfanos fuera del manifest”, pero separado

Eso elimina mucha ambigüedad.

———

### 9. Agent03 debe ser solo reporting/gate

Le quitaría cualquier papel de construir inputs o arreglar artefactos de producción.

Agent03 solo debería leer:

- task_manifest
- task_status
- validation outputs
- lifecycle
- calendario oficial

Y producir:

- cobertura
- causas
- resumen
- gate

Nada más.

———

### 10. Todo relanzamiento debe nacer de manifest derivado, no de notebooks manuales

Flujos permitidos:

- full_run_manifest.parquet
- retry_manifest.parquet
- empty_recheck_manifest.parquet

Cada uno con su hash, origen y fecha.

No haría más CSVs manuales “final”, “v2”, “fixed”, etc.

———

## Flujo limpio propuesto

### Fase A. Build del universo

- build_universe_pti.py
- refine_universe_target.py
- output congelado:
    - universe_snapshot.parquet

### Fase B. Build de tareas

- build_quotes_task_manifest.py
- usa:
    - universe_snapshot.parquet
    - official_lifecycle_compiled.csv
    - calendario oficial XNYS
    - límites del run
- output:
    - tasks_quotes_prod.parquet
    - tasks_quotes_prod_meta.json

### Fase C. Descarga

- download_quotes.py
- input:
    - tasks_quotes_prod.parquet
    - task_status.parquet
- output:
    - parquet final
    - sidecar por archivo
    - actualización transaccional de task_status

### Fase D. Validación

- validate_quotes_run.py
- valida solo archivos del manifest
- actualiza validation_status en task_status

### Fase E. Reconciliación

- reconcile_run_state.py
- output:
    - retry_manifest.parquet
    - reconciliation_report.json

### Fase F. Reporting/Gate

- report_quotes_run.py
- output:
    - coverage_by_ticker
    - causes_by_ticker
    - run_summary
    - gate_status

———

## Qué problemas resuelve esto

Resuelve:

- construcción manual de tasks en notebooks operativos
- mezcla entre universo, descarga, validación y reporting
- reanudación ambigua por CSV mutable
- necesidad de hacer análisis local de cada file tras powerloss
- múltiples CSVs “missing final v2”
- validación fuera de manifest
- scans de disco no acotados al run
- ambigüedad entre “descargado”, “válido”, “vacío”, “confirmado vacío”

———

## Mi recomendación práctica

No intentaría reescribir todo de golpe.

Haría 3 pasos:

1. Mantener download_quotes.py, pero cambiarlo para que escriba task_status.parquet y sidecars.
2. Sacar la construcción de tasks_quotes_prod.* a un script único reproducible.
3. Reemplazar missing_only_final_v2.csv por reconcile_run_state.py -> retry_manifest.parquet.

Con solo esos 3 cambios, el sistema ya pasa de “operativo pero frágil” a “auditably resumable”.

## Sobre tu notebook de auditoría

Tiene sentido. La primera celda debería fijar:

- run_id
- run_dir
- universe sources
- task sources
- state sources
- quotes_root
- hashes o timestamps de artefactos clave

Y la segunda celda sí debería formalizar el contrato esperado de Agent01.

In [7]:
from pathlib import Path
import json
import pandas as pd

CONTRACT_FILE = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml")
CONTRACT_VIEWER = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\030_show_contract_polygon_download.py")

# Test histórico
TEST_RUN_ID = "20260312_quotes_lifecycle3_test"
TEST_RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / TEST_RUN_ID
TEST_QUOTES_ROOT = Path(r"D:\quotes\__pruebas__\lifecycle_prod")
TEST_TASKS_CSV = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3.csv"
TEST_TASKS_META = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3_meta.json"

# Producción actual
PROD_RUN_ID = "20260313_quotes_prod_full_12133_clean"
PROD_RUN_DIR_TEMPLATE = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit")
PROD_RUN_DIR = PROD_RUN_DIR_TEMPLATE / PROD_RUN_ID
PROD_QUOTES_ROOT = Path(r"D:\quotes")

PROD_TASKS_CSV = PROD_RUN_DIR / "inputs" / "tasks_quotes_prod.csv"
PROD_TASKS_MISSING_CSV = PROD_RUN_DIR / "inputs" / "tasks_quotes_prod_missing_only_final_v2.csv"
PROD_CURRENT_CSV = PROD_RUN_DIR / "download_events_current.csv"
PROD_LIVE_STATUS_JSON = PROD_RUN_DIR / "download_live_status.json"
PROD_REPAIRED_CSV = PROD_RUN_DIR / "download_events_current.repaired_from_history.csv"
PROD_DISK_INVENTORY_CSV = PROD_RUN_DIR / "disk_quotes_inventory.csv"

print("CONTRACT_FILE:", CONTRACT_FILE, "exists", CONTRACT_FILE.exists())
print("CONTRACT_VIEWER:", CONTRACT_VIEWER, "exists", CONTRACT_VIEWER.exists())
print()

print("TEST_RUN_DIR:", TEST_RUN_DIR, "exists", TEST_RUN_DIR.exists())
print("TEST_QUOTES_ROOT:", TEST_QUOTES_ROOT, "exists", TEST_QUOTES_ROOT.exists())
print()

print("PROD_RUN_ID:", PROD_RUN_ID)
print("PROD_RUN_DIR:", PROD_RUN_DIR, "exists", PROD_RUN_DIR.exists())
print("PROD_QUOTES_ROOT:", PROD_QUOTES_ROOT, "exists", PROD_QUOTES_ROOT.exists())
print("PROD_TASKS_CSV:", PROD_TASKS_CSV, "exists", PROD_TASKS_CSV.exists())
print("PROD_TASKS_MISSING_CSV:", PROD_TASKS_MISSING_CSV, "exists", PROD_TASKS_MISSING_CSV.exists())
print("PROD_CURRENT_CSV:", PROD_CURRENT_CSV, "exists", PROD_CURRENT_CSV.exists())
print("PROD_LIVE_STATUS_JSON:", PROD_LIVE_STATUS_JSON, "exists", PROD_LIVE_STATUS_JSON.exists())
print("PROD_REPAIRED_CSV:", PROD_REPAIRED_CSV, "exists", PROD_REPAIRED_CSV.exists())
print("PROD_DISK_INVENTORY_CSV:", PROD_DISK_INVENTORY_CSV, "exists", PROD_DISK_INVENTORY_CSV.exists())

CONTRACT_FILE: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml exists True
CONTRACT_VIEWER: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\030_show_contract_polygon_download.py exists True

TEST_RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test exists True
TEST_QUOTES_ROOT: D:\quotes\__pruebas__\lifecycle_prod exists False

PROD_RUN_ID: 20260313_quotes_prod_full_12133_clean
PROD_RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean exists True
PROD_QUOTES_ROOT: D:\quotes exists True
PROD_TASKS_CSV: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\tasks_quotes_prod.csv exists True
PROD_TASKS_MISSING_CSV: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\tasks_quotes_prod_mis

**PROD_RUN_ID**: identificador único de la corrida de producción actual de `quotes`. Todo el estado operativo, los CSV
de seguimiento y los outputs de validación cuelgan de este `RUN_ID`.

**PROD_RUN_DIR**: carpeta raíz del run de producción actual. Aquí se guardan los estados de Agent01, Agent02, Agent03,
los CSV de eventos, los snapshots vivos y los artefactos de reconciliación.

**PROD_QUOTES_ROOT**: raíz física de los parquets de `quotes` en disco. Cada tarea `ticker,date` termina materializada
aquí como `D:\quotes\<TICKER>\year=<YYYY>\month=<MM>\day=<DD>\quotes.parquet`.

**PROD_TASKS_CSV**: contrato maestro del run. Define el universo esperado completo de tareas `ticker,date` que la
corrida debería cubrir.

**PROD_TASKS_MISSING_CSV**: subconjunto derivado del contrato maestro con solo los faltantes reales que todavía
interesa intentar descargar. Es el CSV correcto para relanzar Agent01 sin rehacer todo el universo.

**PROD_CURRENT_CSV**: snapshot operativo actual de Agent01 por tarea. Guarda el estado más reciente conocido de cada
`task_key`, por ejemplo `DOWNLOADED_OK`, `DOWNLOADED_EMPTY` o errores de descarga.

**PROD_LIVE_STATUS_JSON**: resumen agregado vivo de Agent01. Sirve para ver progreso total, conteos por estado,
pendientes y configuración efectiva del downloader sin leer el CSV completo.

**PROD_REPAIRED_CSV**: reconstrucción reparada de `download_events_current` a partir del histórico del run. Se usa
como referencia más fiable cuando el `current` operativo ha quedado degradado o inconsistente.

**PROD_DISK_INVENTORY_CSV**: inventario de los parquets detectados físicamente en `D:\quotes` en una foto concreta del
disco. Sirve para reconciliar “lo que realmente existe” frente a “lo que el estado del run dice que existe”.

---

**Contrato ymal para agenet 02**

In [17]:
#from IPython.display import Markdown, display

#if CONTRACT_FILE.exists():
#    raw = CONTRACT_FILE.read_text(encoding="utf-8")
#    display(Markdown(f"```yaml\n{raw}\n```"))
#else:
#    print("Contrato no disponible")

In [18]:
from IPython.display import Markdown, display

if CONTRACT_FILE.exists():
    raw = CONTRACT_FILE.read_text(encoding="utf-8")
    start = raw.find("quotes:")
    end = raw.find("\ntrades_ticks_2005_2025:", start)

    if start != -1:
        quotes_raw = raw[start:end if end != -1 else None]
        display(Markdown(f"```yaml\n{quotes_raw}\n```"))
    else:
        print("No encontré bloque quotes en el contrato")
else:
    print("Contrato no disponible")

```yaml
quotes:
    enabled: true
    priority: 1
    target_path: "quotes"
    expected_file_name: "quotes.parquet"
    partition_pattern: "{ticker}/year={year}/month={month}/day={day}/quotes.parquet"
    partition_keys: ["ticker", "year", "month", "day"]

    # Minimo para certificar cobertura mientras descarga (modo laxo)
    minimum_required_columns: ["timestamp", "bid_price", "ask_price"]
    non_empty_required: true
    min_rows_per_file: 1

    # Reglas de calidad no bloqueantes
    soft_quality_rules:
      bid_ask_sanity:
        enabled: true
        expression: "bid_price <= ask_price"
        allow_violations: true
      non_negative_prices:
        enabled: true
        columns: ["bid_price", "ask_price"]
        allow_violations: true

    coverage:
      coverage_unit: "day"
      expected_files_per_day: 1
      success_criteria: "file_exists_and_rows_gt_0"
      completeness_metric: "present_days / expected_days_official_window"

    retry:  # legacy section name; en produccion se interpreta como review queue / remediacion posterior
      enabled: true
      backoff_seconds: [30, 120, 600]

  trades_ticks_2005_2025:
    enabled: true
    priority: 2
    target_path: "trades_ticks_2005_2025"
    expected_file_name: "market.parquet"
    partition_pattern: "{ticker}/year={year}/month={month}/day={day}/market.parquet"
    partition_keys: ["ticker", "year", "month", "day"]

    # Minimo para certificar cobertura mientras descarga (modo laxo)
    minimum_required_columns: ["timestamp", "price", "size"]
    non_empty_required: true
    min_rows_per_file: 1

    # Reglas de calidad no bloqueantes
    soft_quality_rules:
      non_negative_price:
        enabled: true
        columns: ["price"]
        allow_violations: true
      positive_size:
        enabled: true
        columns: ["size"]
        allow_violations: true
      timestamp_monotonic:
        enabled: false
        note: "Desactivado en modo laxo para evitar frenar ingesta por desorden puntual"

    coverage:
      coverage_unit: "day"
      expected_files_per_day: 1
      success_criteria: "file_exists_and_rows_gt_0"
      completeness_metric: "present_days / expected_days_official_window"

    retry:  # legacy section name; en produccion se interpreta como review queue / remediacion posterior
      enabled: true
      backoff_seconds: [30, 120, 600]

# Campos esperados desde fuente oficial de lifecycle (para calcular expected_days_official_window)
official_window_source:
  preferred_files:
    - "C:/TSIS_Data/02_backtest_SmallCaps/data/reference/official_lifecycle_compiled.csv"
    - "C:/TSIS_Data/02_backtest_SmallCaps/data/reference/official_ticker_events.csv"
  fallback_mode: "observed_min_max_if_official_missing"

# Convencion de severidad para panel/monitor
severities:
  pass: 0
  pass_with_warn: 1
  soft_fail: 2
  hard_fail: 3

# Comandos sugeridos para terminales (referencia)
terminal_plan:
  terminal_1_downloader: "python run_downloader.py --contract contracts_polygon_download.yaml --dataset quotes"
  terminal_2_validator: "python run_validator_watch.py --contract contracts_polygon_download.yaml"
  terminal_3_stats: "python run_stats_watch.py --contract contracts_polygon_download.yaml"

```